# Imports

In [1]:
%%capture
!pip install segmentation-models-pytorch
!pip install torchinfo

In [2]:
# Data handling
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2

# Torch
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import segmentation_models_pytorch as smp
from torchinfo import summary

# os
import os

# Path
from pathlib import Path

# tqdm
from tqdm.auto import tqdm

from glob import iglob, glob
from itertools import chain

# warnings
import warnings
warnings.filterwarnings("ignore")

import random as rnd

import shutil

In [3]:
BATCH_SIZE = 64
NUM_WORKERS = os.cpu_count()

# Num of samples, that will be used for train out model
K_SAMPLES_TRAIN = 5

NUM_CLASSES = 2

# CUDA
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Dataloaders

## Kaggle API Setup

In [4]:
!pip install -q kaggle

In [5]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"michleat","key":"f02a3525eedb6f27532b72a5bad27a40"}'}

In [6]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

## Download GlaS

In [7]:
!kaggle datasets download -d sani84/glasmiccai2015-gland-segmentation -p /content/glas

Dataset URL: https://www.kaggle.com/datasets/sani84/glasmiccai2015-gland-segmentation
License(s): other
 94% 162M/172M [00:03<00:00, 65.9MB/s]
100% 172M/172M [00:03<00:00, 55.1MB/s]


In [8]:
!unzip /content/glas/glasmiccai2015-gland-segmentation.zip -d /content/glas/

Archive:  /content/glas/glasmiccai2015-gland-segmentation.zip
  inflating: /content/glas/Warwick_QU_Dataset/Grade.csv  
  inflating: /content/glas/Warwick_QU_Dataset/testA_1.bmp  
  inflating: /content/glas/Warwick_QU_Dataset/testA_10.bmp  
  inflating: /content/glas/Warwick_QU_Dataset/testA_10_anno.bmp  
  inflating: /content/glas/Warwick_QU_Dataset/testA_11.bmp  
  inflating: /content/glas/Warwick_QU_Dataset/testA_11_anno.bmp  
  inflating: /content/glas/Warwick_QU_Dataset/testA_12.bmp  
  inflating: /content/glas/Warwick_QU_Dataset/testA_12_anno.bmp  
  inflating: /content/glas/Warwick_QU_Dataset/testA_13.bmp  
  inflating: /content/glas/Warwick_QU_Dataset/testA_13_anno.bmp  
  inflating: /content/glas/Warwick_QU_Dataset/testA_14.bmp  
  inflating: /content/glas/Warwick_QU_Dataset/testA_14_anno.bmp  
  inflating: /content/glas/Warwick_QU_Dataset/testA_15.bmp  
  inflating: /content/glas/Warwick_QU_Dataset/testA_15_anno.bmp  
  inflating: /content/glas/Warwick_QU_Dataset/testA_16.bmp

In [9]:
!rm /content/glas/glasmiccai2015-gland-segmentation.zip

## Defining train for meta-test

In [10]:
!rm /content/glas/Warwick_QU_Dataset/Grade.csv

In [11]:
def get_glas_train():
    rnd.seed(42)
    global K_SAMPLES_TRAIN

    files = glob("/content/glas/Warwick_QU_Dataset/*.bmp")
    train_labels = rnd.sample(glob("/content/glas/Warwick_QU_Dataset/*anno.bmp"), k=K_SAMPLES_TRAIN)
    train_files = [p for p in glob("/content/glas/Warwick_QU_Dataset/*.bmp") if p.replace(".bmp", "_anno.bmp") in train_labels]

    return train_files + train_labels

In [12]:
!mkdir /content/glas/Warwick_QU_Dataset/train
!mkdir /content/glas/Warwick_QU_Dataset/test

In [13]:
train_files = get_glas_train()

for f in iglob("/content/glas/Warwick_QU_Dataset/*.bmp"):
    if f in train_files:
        shutil.move(f, f.replace("/content/glas/Warwick_QU_Dataset", "/content/glas/Warwick_QU_Dataset/train"))
    else:
        shutil.move(f, f.replace("/content/glas/Warwick_QU_Dataset", "/content/glas/Warwick_QU_Dataset/test"))

# Data prepare

In [14]:
!rm -rf /content/glas/Warwick_QU_Dataset/train/images
!rm -rf /content/glas/Warwick_QU_Dataset/train/labels

!rm -rf /content/glas/Warwick_QU_Dataset/test/images
!rm -rf /content/glas/Warwick_QU_Dataset/test/labels

In [15]:
!mkdir -p /content/glas/Warwick_QU_Dataset/train/images
!mkdir -p /content/glas/Warwick_QU_Dataset/train/labels

!mkdir -p /content/glas/Warwick_QU_Dataset/test/images
!mkdir -p /content/glas/Warwick_QU_Dataset/test/labels

In [16]:
for p in Path("/content/glas/Warwick_QU_Dataset/train/").glob("*anno.bmp"):
    shutil.move(p, "/content/glas/Warwick_QU_Dataset/train/labels/")

for p in Path("/content/glas/Warwick_QU_Dataset/train/").glob("*.bmp"):
    shutil.move(p, "/content/glas/Warwick_QU_Dataset/train/images")

for p in Path("/content/glas/Warwick_QU_Dataset/test/").glob("*anno.bmp"):
    shutil.move(p, "/content/glas/Warwick_QU_Dataset/test/labels/")

for p in Path("/content/glas/Warwick_QU_Dataset/test/").glob("*.bmp"):
    shutil.move(p, "/content/glas/Warwick_QU_Dataset/test/images")

In [17]:
for p in Path("/content/glas/Warwick_QU_Dataset/train/labels/").glob("*.bmp"):
    os.rename(p, p.parent / (p.name.replace("_anno", "")))

for p in Path("/content/glas/Warwick_QU_Dataset/test/labels/").glob("*.bmp"):
    os.rename(p, p.parent / (p.name.replace("_anno", "")))

In [18]:
for p in Path("/content/glas/Warwick_QU_Dataset/train/labels/").glob("*.bmp"):
    img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    bin_mask = np.zeros(img.shape)
    bin_mask[img != 0] = 1
    cv2.imwrite(str(p), bin_mask)

In [19]:
for p in Path("/content/glas/Warwick_QU_Dataset/test/labels/").glob("*.bmp"):
    img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    bin_mask = np.zeros(img.shape)
    bin_mask[img != 0] = 1
    cv2.imwrite(str(p), bin_mask)

# Utils

## Data

In [20]:
def image_mask_path(image_path: str, mask_path: str):
    IMAGE_PATH = Path(image_path)
    IMAGE_PATH_LIST = sorted(list(IMAGE_PATH.glob("*.bmp")))

    MASK_PATH = Path(mask_path)
    MASK_PATH_LIST = sorted(list(MASK_PATH.glob("*.bmp")))

    return IMAGE_PATH_LIST, MASK_PATH_LIST

In [21]:
def count_unique(mask_path_list):
    VALUES_UNIQUE_TRAIN = []

    for i in mask_path_list:
        sample = cv2.imread(str(i), cv2.IMREAD_GRAYSCALE)
        uniques = np.unique(sample)
        VALUES_UNIQUE_TRAIN.append(uniques)

    FINAL_VALUES_UNIQUE_TRAIN = np.concatenate(VALUES_UNIQUE_TRAIN)

    return np.unique(FINAL_VALUES_UNIQUE_TRAIN)

In [22]:
class CustomImageMaskDataset(Dataset):
    def __init__(self, data:pd.DataFrame, image_transforms, mask_transforms):
        self.data = data
        self.image_transforms = image_transforms
        self.mask_transforms = mask_transforms

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image_path = self.data.iloc[idx, 0]
        image = Image.open(image_path).convert("RGB")

        state = torch.get_rng_state()
        image = self.image_transforms(image)

        mask_path = self.data.iloc[idx, 1]
        mask = Image.open(mask_path)

        torch.set_rng_state(state)
        mask = self.mask_transforms(mask)

        return image, mask

In [23]:
class CustomTestDataset(Dataset):
    def __init__(self, data:pd.DataFrame, image_transforms):
        self.data = data
        self.image_transforms = image_transforms

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image_path = self.data.iloc[idx, 0]
        image = Image.open(image_path).convert("RGB")
        image = self.image_transforms(image)

        return image

## Train

In [24]:
def train_step(model:torch.nn.Module, dataloader:torch.utils.data.DataLoader,
               loss_fn:torch.nn.Module, optimizer:torch.optim.Optimizer):

    model.train()

    train_loss = 0.
    train_dice = 0.

    for batch, (X,y) in enumerate(dataloader):
        X = X.to(device = DEVICE, dtype = torch.float32)
        y = y.to(device = DEVICE, dtype = torch.long)
        optimizer.zero_grad()
        logit_mask = model(X)
        loss = loss_fn(logit_mask, y.squeeze())
        train_loss += loss.item()

        loss.backward()
        optimizer.step()

        prob_mask = logit_mask.softmax(dim = 1)
        pred_mask = prob_mask.argmax(dim = 1)

        tp,fp,fn,tn = smp.metrics.get_stats(output = pred_mask.detach().cpu().long(),
                                            target = y.squeeze().cpu().long(),
                                            mode = "multiclass",
                                            num_classes = 21)

        train_dice += smp.metrics.f1_score(tp, fp, fn, tn, reduction = "micro").numpy()

    train_loss = train_loss / len(dataloader)
    train_dice = train_dice / len(dataloader)

    return train_loss, train_dice

In [25]:
def train(model:torch.nn.Module, train_dataloader:torch.utils.data.DataLoader,
          loss_fn:torch.nn.Module,
          optimizer:torch.optim.Optimizer, epochs:int = 10):

    results = {'train_loss':[], 'train_dice':[]}

    for epoch in tqdm(range(epochs)):
        train_loss, train_dice = train_step(model = model,
                                           dataloader = train_dataloader,
                                           loss_fn = loss_fn,
                                           optimizer = optimizer)

        print(f'Epoch: {epoch + 1} | ',
              f'Train Loss: {train_loss:.4f} | ',
              f'Train Dice: {train_dice:.4f}')

        results['train_loss'].append(train_loss)
        results['train_dice'].append(train_dice)

    return results

## Prediction

In [26]:
def predictions_mask(model, test_dataloader: torch.utils.data.DataLoader):
    model.eval()

    y_pred_mask = []

    with torch.inference_mode():
        for batch,X in enumerate(test_dataloader):
            X = X.to(device = DEVICE, dtype = torch.float32)
            mask_logit = model(X)
            mask_prob = mask_logit.softmax(dim = 1)
            mask_pred = mask_prob.argmax(dim = 1)
            y_pred_mask.append(mask_pred.detach().cpu())

    y_pred_mask = torch.cat(y_pred_mask)

    return y_pred_mask

# Augmentations

In [27]:
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

image_transforms = transforms.Compose([
                                       transforms.RandomHorizontalFlip(),
                                       transforms.RandomVerticalFlip(),
                                       transforms.RandomResizedCrop((224, 224), antialias=True),
                                       transforms.ToTensor(),
                                       transforms.Normalize(mean = MEAN, std = STD),
                                       ])

mask_transforms = transforms.Compose([
                                      transforms.RandomHorizontalFlip(),
                                      transforms.RandomVerticalFlip(),
                                      transforms.RandomResizedCrop((224, 224), antialias=True),
                                      transforms.PILToTensor(),
                                    ])

image_transforms_test = transforms.Compose([
                                       transforms.Resize((224, 224), antialias=True),
                                       transforms.ToTensor(),
                                       transforms.Normalize(mean = MEAN, std = STD),
                                       ])

mask_transforms_test = transforms.Compose([
                                      transforms.Resize((224, 224), antialias=True),
                                      transforms.PILToTensor(),
                                    ])

# Data load

In [28]:
image_path_train = "/content/glas/Warwick_QU_Dataset/train/images"
mask_path_train = "/content/glas/Warwick_QU_Dataset/train/labels"

IMAGE_PATH_LIST_TRAIN, MASK_PATH_LIST_TRAIN = image_mask_path(image_path_train,
                                                              mask_path_train)

print(f'Total Images Train: {len(IMAGE_PATH_LIST_TRAIN)}')
print(f'Total Masks Train: {len(MASK_PATH_LIST_TRAIN)}')

Total Images Train: 5
Total Masks Train: 5


In [29]:
print("Unique values Train:")
print(count_unique(MASK_PATH_LIST_TRAIN))

Unique values Train:
[0 1]


# Preprocessing

In [30]:
data_train = pd.DataFrame({'Image':IMAGE_PATH_LIST_TRAIN, 'Mask': MASK_PATH_LIST_TRAIN})

In [31]:
train_dataset = CustomImageMaskDataset(data_train, image_transforms, mask_transforms)

In [32]:
train_dataloader = DataLoader(dataset = train_dataset, batch_size = BATCH_SIZE,
                              shuffle = True, num_workers = NUM_WORKERS)

In [33]:
# We visualize the dimensions of a batch.
batch_images, batch_masks = next(iter(train_dataloader))

batch_images.shape, batch_masks.shape

(torch.Size([5, 3, 224, 224]), torch.Size([5, 1, 224, 224]))

# Model

## Load from checkpoint

In [34]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


You need to add link to this pretrained model ([GDrive](https://drive.google.com/file/d/1-QXEPwwszMdM8LB7FHutwreFWsG4Qpn2/view?usp=drive_link))

In [35]:
model = torch.load("/content/drive/MyDrive/biocad_cis/model.pt", map_location=torch.device('cpu'))

## Freeze layers

In [36]:
model.segmentation_head[0].out_channels = NUM_CLASSES

In [37]:
for param in model.encoder.parameters():
    param.requires_grad = False

In [38]:
# We view our model again to check if the encoder layers freeze.
summary(model = model,
        input_size = [64, 3, 224, 224],
        col_width = 15,
        col_names = ['input_size', 'output_size', 'num_params', 'trainable'],
        row_settings = ['var_names'])

Layer (type (var_name))                            Input Shape     Output Shape    Param #         Trainable
Unet (Unet)                                        [64, 3, 224, 224] [64, 3, 224, 224] --              Partial
├─ResNetEncoder (encoder)                          [64, 3, 224, 224] [64, 3, 224, 224] --              False
│    └─Conv2d (conv1)                              [64, 3, 224, 224] [64, 64, 112, 112] (9,408)         False
│    └─BatchNorm2d (bn1)                           [64, 64, 112, 112] [64, 64, 112, 112] (128)           False
│    └─ReLU (relu)                                 [64, 64, 112, 112] [64, 64, 112, 112] --              --
│    └─MaxPool2d (maxpool)                         [64, 64, 112, 112] [64, 64, 56, 56] --              --
│    └─Sequential (layer1)                         [64, 64, 56, 56] [64, 64, 56, 56] --              False
│    │    └─BasicBlock (0)                         [64, 64, 56, 56] [64, 64, 56, 56] (73,984)        False
│    │    └─BasicBlock

## Train

In [39]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001, weight_decay = 0.0001)

In [40]:
# Training!!!

SEED = 42
EPOCHS = 10
torch.cuda.manual_seed(SEED)
torch.manual_seed(SEED)

RESULTS = train(model.to(device = DEVICE),
                train_dataloader,
                loss_fn,
                optimizer,
                EPOCHS)

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch: 1 |  Train Loss: 1.2842 |  Train Dice: 0.5020
Epoch: 2 |  Train Loss: 1.3250 |  Train Dice: 0.4341
Epoch: 3 |  Train Loss: 1.2223 |  Train Dice: 0.5125
Epoch: 4 |  Train Loss: 1.0311 |  Train Dice: 0.5351
Epoch: 5 |  Train Loss: 0.9278 |  Train Dice: 0.5438
Epoch: 6 |  Train Loss: 0.7486 |  Train Dice: 0.6234
Epoch: 7 |  Train Loss: 0.7105 |  Train Dice: 0.6977
Epoch: 8 |  Train Loss: 0.6332 |  Train Dice: 0.7063
Epoch: 9 |  Train Loss: 0.6703 |  Train Dice: 0.6350
Epoch: 10 |  Train Loss: 0.5925 |  Train Dice: 0.7324


## Save

In [41]:
!mkdir /content/checkpoints

In [42]:
torch.save(model.state_dict(), "/content/checkpoints/model.pth")

# Evaluation

In [43]:
image_path_val = "/content/glas/Warwick_QU_Dataset/test/images"
mask_path_val = "/content/glas/Warwick_QU_Dataset/test/labels"

IMAGE_PATH_LIST_VAL, MASK_PATH_LIST_VAL = image_mask_path(image_path_val,
                                                          mask_path_val)

print(f'Total Images Val: {len(IMAGE_PATH_LIST_VAL)}')
print(f'Total Masks Val: {len(MASK_PATH_LIST_VAL)}')

Total Images Val: 160
Total Masks Val: 160


In [44]:
data_val = pd.DataFrame({'Image':IMAGE_PATH_LIST_VAL,
                         'Mask':MASK_PATH_LIST_VAL})
val_dataset = CustomImageMaskDataset(data_val, image_transforms_test,
                                     mask_transforms_test)
val_dataloader = DataLoader(dataset = val_dataset, batch_size = BATCH_SIZE,
                            shuffle = True, num_workers = NUM_WORKERS)

In [47]:
# Num of batches, which will be used for evaluation
BATCH_TO_TEST = min(30, len(val_dataloader))

In [48]:
test_dice = 0.

with torch.inference_mode():
    for batch, (X, y) in tqdm(enumerate(val_dataloader), total=BATCH_TO_TEST):
        if batch >= BATCH_TO_TEST:
            break

        X = X.to(device = DEVICE, dtype = torch.float32)
        y = y.to(device = DEVICE, dtype = torch.long)

        logit_mask = model(X)

        prob_mask = logit_mask.softmax(dim = 1)
        pred_mask = prob_mask.argmax(dim = 1)

        tp, fp, fn, tn = smp.metrics.get_stats(output = pred_mask.detach().cpu().long(),
                                                target = y.squeeze().cpu().long(),
                                                mode = "multiclass",
                                                num_classes = 3)

        test_dice += smp.metrics.f1_score(tp, fp, fn, tn, reduction = "micro").numpy()

test_dice /= BATCH_TO_TEST

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a669e636170>
Traceback (most recent call last):
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a669e636170>


  0%|          | 0/3 [00:00<?, ?it/s]

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py", line 1479, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py", line 1479, in __del__
  File "/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py", line 1462, in _shutdown_workers
        if w.is_alive():
  File "/usr/lib/python3.10/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
self._shutdown_workers()AssertionError
: can only test a child process
  File "/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py", line 1462, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.10/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process


In [49]:
test_dice

0.6318385203679403